# P4 recovery — B. Fresh Colab T4
Prerequisite: notebook A sealed checkpoints in the previous runtime, then that runtime was **deleted**.
Choose T4 GPU. Upload the same recovery ZIP. Same-VM checks HOLD before dependency/model loading.

P4 scorer-ready is already closed PASS. This notebook never runs new generation, metrics, fixtures, pilot40,
selection300 or test. It replays all123generation nodes/21outputs and21scorer checkpoints using original keys.
Generation model reload independently verifies original runtime identity; missing cache fails without fallback.
Scorer replay reads archived valid results only. Original cost and fresh load/restore overhead are separate.


In [1]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from google.colab import drive, files
SESSION_STARTED=time.monotonic()
SESSION_LIMIT_SECONDS=11.5*3600
EXPECTED_ZIP_SHA256='53ee5b56a8227d41d0c1b5a651753b11f70dcf45241849c24988b14a548202ad'
EXPECTED_MANIFEST_SHA256='e324a107b2abafe3dd7ddbc087ddd1a02c839455538da99e44c83ab4b1fdbd8f'
WORK=Path('/content/TIB_P4_RECOVERY_261020_V1')
PACKAGE=WORK/'package'
PACKAGE.mkdir(parents=True,exist_ok=True)
drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
STORE=DRIVE/'TIB_P4_RECOVERY_261020_V1'
STORE.mkdir(exist_ok=True)
WITNESS=STORE/'origin_witness.json'
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()
uploaded=files.upload()
archives=[Path(name) for name in uploaded if name.endswith('.zip')]
assert len(archives)==1,'Upload exactly the supplied recovery ZIP'
archive=archives[0]
assert sha(archive)==EXPECTED_ZIP_SHA256,'Wrong recovery ZIP'
with zipfile.ZipFile(archive) as z:
    names=z.namelist()
    assert len(names)==len(set(names)),'Duplicate members'
    for item in z.infolist():
        assert (PACKAGE/item.filename).resolve().is_relative_to(PACKAGE.resolve()),'Unsafe path'
        assert (item.external_attr>>16)&0o170000 != 0o120000,'Symlink member'
    z.extractall(PACKAGE)
assert sha(PACKAGE/'recovery_manifest.json')==EXPECTED_MANIFEST_SHA256
manifest=json.loads((PACKAGE/'recovery_manifest.json').read_text())
assert set(names)==set(manifest['files'])|{'recovery_manifest.json'}
for name,expected in manifest['files'].items():
    assert sha(PACKAGE/name)==expected,name
sys.path.insert(0,str(PACKAGE/'tools'))
from tib_p4_session_recovery import seal_origin, verify_fresh
from tib_p4_recovery_context import create_run_context, finalize
print('Recovery package verified; original stores will be read-only.')


Mounted at /content/drive


Saving tib-p4-fresh-recovery-261020-v1.zip to tib-p4-fresh-recovery-261020-v1.zip
Recovery package verified; original stores will be read-only.


In [2]:
RECOVERY_PREFLIGHT_OK=False
proof=verify_fresh(PACKAGE,DRIVE,WITNESS)
OUTPUT=STORE/'runs'/('fresh-'+str(time.time_ns()))
context=create_run_context(PACKAGE,DRIVE,WITNESS,OUTPUT)
CONTEXT=OUTPUT/'session_context.json'
print(json.dumps(proof,indent=2))
print('New immutable evidence run:',OUTPUT)
RECOVERY_PREFLIGHT_OK=True
def bootstrap_hold(reason):
    hold={'status':'HOLD','reason':str(reason),'cross_session_recovery_verified':False}
    (OUTPUT/('bootstrap-hold-'+str(time.time_ns())+'.json')).write_text(json.dumps(hold,indent=2))


{
  "state": "FRESH_VM_AND_DRIVE_HASHES_VERIFIED",
  "origin_witness_sha256": "08c65341bd544044f87a28b14a59adeb0ad7bebef10174f5f5a01da3802d02fe",
  "origin_session": {
    "google_colab_imported": true,
    "boot_id": "d51e0a61-f44d-4fa3-8f17-2ee6aa60e72e",
    "pid": 1043,
    "observed_epoch_ns": 1792461642087367553,
    "python": "3.13.15",
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.39"
  },
  "recovery_session": {
    "google_colab_imported": true,
    "boot_id": "9d3bfc45-1fd7-4e67-96b0-d9567a0b7d99",
    "pid": 1013,
    "observed_epoch_ns": 1792461988080590695,
    "python": "3.13.15",
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.39"
  },
  "different_vm_boot_id": true,
  "source_stores_unchanged": true
}
New immutable evidence run: /content/drive/MyDrive/TIB_P4_RECOVERY_261020_V1/runs/fresh-1792462266028208129


## Pinned runtime setup
Uses the tested pip-less venv bootstrap. Direct recovery requirements preserve both original runtime profiles;
the accelerator pin1.15.0 is the same observed generation/scorer version. Any full-runtime mismatch HOLDs.
Recovery needs working disk for the three original model snapshots;32GiB free is an operational guard,
not a full-study storage budget. All subprocesses share the notebook11.5h session deadline.


In [3]:
assert RECOVERY_PREFLIGHT_OK
assert shutil.disk_usage('/content').free >= 32*1024**3,'Need 32 GiB free working disk for recovery assets/setup'
BUNDLE=PACKAGE/'scorer_bundle'
ENV = WORK/'venv'
PYTHON = ENV/'bin/python'
RUN_ENV = dict(os.environ, USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false',
               PYTHONPATH=str(BUNDLE), HF_HUB_DISABLE_TELEMETRY='1')
assert shutil.disk_usage('/content').free >= 12*1024**3, 'Need at least 12 GiB working disk before setup.'
def execute(command, log_name):
    remaining = SESSION_LIMIT_SECONDS-(time.monotonic()-SESSION_STARTED)
    assert remaining>0, 'Session guard reached'
    log_path = OUTPUT/log_name
    try:
        with log_path.open('w') as log:
            result = subprocess.run(command, cwd=BUNDLE, env=RUN_ENV, stdout=log,
                                    stderr=subprocess.STDOUT, timeout=remaining)
        print('exit', result.returncode, '|', log_path)
        print(log_path.read_text(errors='replace')[-5000:])
        return result.returncode
    except Exception as error:
        bootstrap_hold(error)
        raise
# An unsuccessful ensurepip step can leave bin/python present. Always repair the
# venv structure without deleting it or bootstrapping pip inside the target.
created = execute([sys.executable,'-m','venv','--system-site-packages','--without-pip',str(ENV)],
                  'create-venv.log')
if created:
    bootstrap_hold('venv --without-pip failed; see create-venv.log')
assert created==0, 'HOLD: venv creation failed. Export evidence using the last cell.'
probe = "import json,sys; from pathlib import Path; assert sys.prefix != sys.base_prefix; assert Path(sys.prefix).absolute() == Path(sys.argv[1]).absolute(); print(json.dumps({'executable':sys.executable,'prefix':sys.prefix,'base_prefix':sys.base_prefix,'python':sys.version}))"
verified = execute([str(PYTHON),'-c',probe,str(ENV)],'venv-identity.log')
if verified:
    bootstrap_hold('Target interpreter is not the intended venv; see venv-identity.log')
assert verified==0, 'HOLD: target venv identity failed.'
# pip >=22.3 officially supports installing into a pip-less target interpreter.
# Use the base installer only as an installer, never as the scorer interpreter.
setup = execute([sys.executable,'-m','pip','--python',str(PYTHON),'install',
                 '-r',str(PACKAGE/'requirements-recovery.txt')], 'install.log')
if setup:
    bootstrap_hold('Pinned dependency installation failed; see install.log (base pip needs --python support)')
assert setup==0, 'HOLD: dependency setup failed. Export evidence using the last cell.'

DEPENDENCIES_READY=True


exit 0 | /content/drive/MyDrive/TIB_P4_RECOVERY_261020_V1/runs/fresh-1792462266028208129/create-venv.log

exit 0 | /content/drive/MyDrive/TIB_P4_RECOVERY_261020_V1/runs/fresh-1792462266028208129/venv-identity.log
{"executable": "/content/TIB_P4_RECOVERY_261020_V1/venv/bin/python", "prefix": "/content/TIB_P4_RECOVERY_261020_V1/venv", "base_prefix": "/usr", "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]"}

exit 0 | /content/drive/MyDrive/TIB_P4_RECOVERY_261020_V1/runs/fresh-1792462266028208129/install.log
RECOVERY_261020_V1/package/requirements-recovery.txt (line 6)) (3.0.4)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━

## Generation checkpoint recovery
Loads each original generation model once, then recomputes native source windows and reads original nodes.
All7historical profiles are replayed for integrity only; this does not reopen config search.
No missing-node inference is allowed. New generation calls must stay0. Model reload/download can take time.


In [4]:
assert RECOVERY_PREFLIGHT_OK and DEPENDENCIES_READY
RUN_ENV['PYTHONPATH']=str(PACKAGE/'generation_bundle')
GEN_STORE=DRIVE/'TIB_P41_DEVELOPMENT_261019/checkpoints'
remaining=(SESSION_LIMIT_SECONDS-(time.monotonic()-SESSION_STARTED))/3600
gen_command=[str(PYTHON),str(PACKAGE/'tools/tib_p4_generation_recovery.py'),
    '--bundle',str(PACKAGE/'generation_bundle'),'--store',str(GEN_STORE),
    '--expected',str(PACKAGE/'expected_generation.json'),'--output',str(OUTPUT/'generation'),
    '--session-context',str(CONTEXT),'--wall-hours',str(min(11.5,remaining))]
generation_exit=execute(gen_command,'generation-subprocess.log')
print('Generation exit:',generation_exit,'(inspect receipt; do not retry by inference)')


exit 1 | /content/drive/MyDrive/TIB_P4_RECOVERY_261020_V1/runs/fresh-1792462266028208129/generation-subprocess.log
Original store and generation bundle hashes verified
Loading original pinned assets: bart

Fetching 7 files: 100%|██████████| 7/7 [00:23<00:00,  3.30s/it]
`torch_dtype` is deprecated! Use `dtype` instead!
ValueError: Original loaded configuration differs: resolved_model_config
Recovery status: HOLD

Generation exit: 1 (inspect receipt; do not retry by inference)


## Scorer cache replay
Reads the original21Drive checkpoints with exact original runtime/input/key identity; no scorer model loads,
metric calls or fixture reruns. A changed full package inventory, Python/platform or thread configuration HOLDs.
Do not edit identities/checkpoint keys to make the current runtime match.


In [5]:
assert RECOVERY_PREFLIGHT_OK and DEPENDENCIES_READY
RUN_ENV['PYTHONPATH']=str(PACKAGE/'scorer_bundle')
score_command=[str(PYTHON),str(PACKAGE/'tools/tib_p4_scorer_replay.py'),
    '--bundle',str(PACKAGE/'scorer_bundle'),
    '--store',str(DRIVE/'TIB_P4_SCORER_261020_V1/scorer-evidence/checkpoints'),
    '--origin-ready',str(PACKAGE/'origin_scorer_ready.json'),
    '--asset-manifests',str(PACKAGE/'scorer_asset_manifests'),
    '--expected',str(PACKAGE/'expected_scorer.json'),'--output',str(OUTPUT/'scorer'),
    '--session-context',str(CONTEXT)]
scorer_exit=execute(score_command,'scorer-replay-subprocess.log')
print('Scorer replay exit:',scorer_exit)


exit 0 | /content/drive/MyDrive/TIB_P4_RECOVERY_261020_V1/runs/fresh-1792462266028208129/scorer-replay-subprocess.log
{"status": "PASS", "reused_records": 21}

Scorer replay exit: 0


## Combine and export evidence
Combined PASS requires both actual stage results, this VM/run/witness bindings,123/21generation reuse,
21×5finite cached scores, zero new calls and unchanged original stores. Same-session/local tests cannot certify it.
Keep the ZIP private. No model weight files or virtual environment are exported. Pilot40 remains a separate gate.


In [6]:
result=finalize(PACKAGE,DRIVE,WITNESS,OUTPUT)
print(json.dumps({'status':result['status'],'reasons':result['reasons'],
                  'cross_session_recovery_verified':result['cross_session_recovery_verified'],
                  'new_generation_calls':result['new_generation_calls'],
                  'new_metric_calls':result['new_metric_calls']},indent=2))


{
  "status": "HOLD",
  "reasons": [
    "generation_stage_identity_counts_or_zero_call_proof_failed"
  ],
  "cross_session_recovery_verified": false,
  "new_generation_calls": 0,
  "new_metric_calls": 0
}


In [7]:
from datetime import datetime,timezone
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
export=STORE/('tib-p4-recovery-evidence-'+stamp+'.zip')
with zipfile.ZipFile(export,'w',zipfile.ZIP_DEFLATED) as z:
    for root,prefix in [(OUTPUT,'recovery'),(PACKAGE,'package')]:
        for path in sorted(root.rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts:
                z.write(path,prefix+'/'+path.relative_to(root).as_posix())
    z.write(WITNESS,'origin_witness.json')
print(export,'bytes',export.stat().st_size,'SHA256',sha(export))
files.download(str(export))


/content/drive/MyDrive/TIB_P4_RECOVERY_261020_V1/tib-p4-recovery-evidence-20261020T021235Z.zip bytes 543837 SHA256 9fe7936ff078e4c556e2155314f8f9b17e1f0a1a93328e5d44ab455cead4fa9c


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>